# Extraction Analysis

Recovery (recall/precision vs. ground truth) and hallucination rate for a single extraction run.

In [1]:
import sys
from pathlib import Path
%load_ext autoreload
%autoreload 2

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'experiments'))
sys.path.insert(0, str(REPO_ROOT))

In [2]:
# ── Config ────────────────────────────────────────────────────────────────────
DATASET         = 'nfix'
MODEL           = 'gemma-3-27b'
EXTRACTION_DATE = '2026_04_20'  # set to None to use most recent run

In [9]:
import pandas as pd
from analysis.loaders import load_extraction, load_combined_judgements, load_ground_truth, cached_match
from scholarlm.utils.unit_conversion import apply_unit_conversion
from analysis.metrics import recovery_rate, hallucination_rate, per_paper_metrics
from analysis.plots import recovery_bar, probability_distribution
from experiments.run_extraction import load_dataset_config
import paths

config = load_dataset_config(DATASET)

## Load data

In [4]:
records = load_extraction(DATASET, MODEL, EXTRACTION_DATE)
extraction_df = pd.DataFrame(records)
extraction_df = apply_unit_conversion(extraction_df, config.unit_conversion_table)
print(f'{len(extraction_df)} extracted measurements')
extraction_df.head()

5680 extracted measurements


,reference,doi,doi_data,url,publication_year,authors,title,publication,volume,pages,...,value,units,page_number,table_number,row_index,column_index,source,context,measurement_id,converted_value
0,Hicks and Silvester 1990 N Zealand J Mar Fresh...,10.1080/00288330.1990.9516439,NaN,http://www.tandfonline.com/doi/abs/10.1080/002...,1990,"Hicks, Brendan J.; Silvester, Warwick B.",Acetylene reduction associated with Zostera no...,New Zealand Journal of Marine and Freshwater R...,24,481-486,...,18,umol C₂H₄ m⁻² h⁻¹,[6],[None],[None],[None],[text],"[McClung, C. R.; Patriquin, D. G.; Davis, R. E...",0,18.000
1,Huang et al 2021 Catena 206,10.1016/j.catena.2021.105545,NaN,https://linkinghub.elsevier.com/retrieve/pii/S...,2021,"Huang, Fangjuan; Lin, Xianbiao; Hu, Weifang; Z...",Nitrogen cycling processes in sediments of the...,CATENA,206,105545,...,0.31,μg N g−¹ dry d−¹,[0],[None],[None],[None],[text],[Nitrogen cycling processes in sediments of th...,1,0.310
2,Huang et al 2021 Catena 206,10.1016/j.catena.2021.105545,NaN,https://linkinghub.elsevier.com/retrieve/pii/S...,2021,"Huang, Fangjuan; Lin, Xianbiao; Hu, Weifang; Z...",Nitrogen cycling processes in sediments of the...,CATENA,206,105545,...,4.43 × 10⁶,copies g⁻¹ sediment,[5],[None],[None],[None],[text],[Fig. 3. The spatial variations of sediment N ...,2,NaN
3,Huang et al 2021 Catena 206,10.1016/j.catena.2021.105545,NaN,https://linkinghub.elsevier.com/retrieve/pii/S...,2021,"Huang, Fangjuan; Lin, Xianbiao; Hu, Weifang; Z...",Nitrogen cycling processes in sediments of the...,CATENA,206,105545,...,2.025,μg N g⁻¹,[7],[None],[None],[None],[text],[Fig. 6. Sediment physicochemical properties (...,3,2.025
4,Huang et al 2021 Catena 206,10.1016/j.catena.2021.105545,NaN,https://linkinghub.elsevier.com/retrieve/pii/S...,2021,"Huang, Fangjuan; Lin, Xianbiao; Hu, Weifang; Z...",Nitrogen cycling processes in sediments of the...,CATENA,206,105545,...,0.31,μg N g−1 dry d−1,[11],[None],[None],[None],[text],[large gap is still remaining in determining i...,4,0.310


In [5]:
ground_truth_df = load_ground_truth(config)
print(f'{len(ground_truth_df)} ground truth measurements')
ground_truth_df.head()

991 ground truth measurements


,nfix_rate_id,reference_id,site_name,latitude,longitude,habitat,year,month,day,hour_minute,season,substrate,substrate_details,attribute,value,error,error_type,units
0,N1,R231,south west coast of Australia,-28.00,114.00,continental shelves,2003.0,10.0,NaN,NaN,NaN,wc,water,nfix_rate,0.017,0.006,stdev.,nmol-n l-1 h-1
1,N2,R231,south west coast of Australia,-28.00,114.00,continental shelves,2003.0,10.0,NaN,NaN,NaN,wc,water,nfix_rate,0.003,0.002,stdev.,nmol-n l-1 h-1
2,N3,R231,south west coast of Australia,-28.00,114.00,continental shelves,2003.0,10.0,NaN,NaN,NaN,wc,water,nfix_rate,0.007,0.006,stdev.,nmol-n l-1 h-1
3,N4,R231,south west coast of Australia,-28.00,114.00,continental shelves,2003.0,10.0,NaN,NaN,NaN,wc,water,nfix_rate,0.001,0.001,stdev.,nmol-n l-1 h-1
4,N5,R147,Arafura and Timor Sea,-10.44,122.16,continental shelves,2012.0,10.0,NaN,NaN,NaN,wc,water,nfix_rate,23.000,32.000,stdev.,nmol-n l-1 d-1


In [ ]:
judgements = load_combined_judgements(DATASET, MODEL, EXTRACTION_DATE)
judged_df = pd.DataFrame(judgements)
print(f'{len(judged_df)} judged measurements')

## Recovery

In [8]:
if 'pond' in DATASET:
    STRICT_MATCHING = {'title': 'title', 'attribute': 'attribute', 'value': 'converted_value'}
    #STRICT_MATCHING = {'title': 'title', 'attribute': 'attribute', 'processed_value': 'value'}
    FUZZY_MATCHING  = {'name': 'name', 'location': 'location', 'ecosystem': 'ecosystem'}
elif 'nfix' in DATASET:
    extraction_df['attribute'] = extraction_df['attribute'].map({'nfix_rate_areal': 'nfix_rate', 'nfix_rate_volumetric': 'nfix_rate', 'nfix_rate_mass': 'nfix_rate', 'nfix_rate':'nfix_rate'})
    STRICT_MATCHING = {'reference_id': 'paper_code', 'attribute': 'attribute', 'value': 'converted_value'}
    FUZZY_MATCHING  = {"site_name": "name", "habitat": "site_type"}
else:
    raise ValueError("Dataset not recognized.")


cache_path = paths.extraction(DATASET, MODEL, EXTRACTION_DATE) / 'match_cache.pkl'
#cache_path = None

matching, edges, weights = cached_match(
    ground_truth_df, extraction_df,
    strict_matching=STRICT_MATCHING,
    fuzzy_matching=FUZZY_MATCHING,
    fuzzy_threshold=0.0,
    cache_path=cache_path,
)

NameError: name 'cached_match' is not defined

In [12]:
# Update strict_matching to match your dataset's entity key columns
if 'pond' in DATASET:
    STRICT_MATCHING = {'title': 'title', 'attribute': 'attribute', 'converted_value': 'value'}
    FUZZY_MATCHING  = {'name': 'name', 'location': 'location', 'ecosystem': 'ecosystem'}
elif 'nfix' in DATASET:
    extraction_df['attribute'] = extraction_df['attribute'].map({'nfix_rate_areal': 'nfix_rate', 'nfix_rate_volumetric': 'nfix_rate', 'nfix_rate_mass': 'nfix_rate'})
    STRICT_MATCHING = {'paper_code': 'reference_id', 'attribute': 'attribute', 'converted_value': 'value'}
    FUZZY_MATCHING  = {"name": "site_name", "site_type": "habitat"}

stats = recovery_rate(
    ground_truth_df,
    extraction_df,
    strict_matching=STRICT_MATCHING,
    fuzzy_matching=FUZZY_MATCHING,
    cache_path=Path(f"../data/experiments/{DATASET}/extraction/{MODEL}/{EXTRACTION_DATE}/match_cache.pkl")
)
print(stats)

{'recovery': 0.4026392961876833, 'precision': 0.061503314818132954, 'n_extracted': 22324, 'n_gt': 3410}


## Hallucination rate

In [ ]:
hall = hallucination_rate(judged_df)
print(hall)

fig = probability_distribution(judged_df, prob_col='judgement_p_true', label_col='judgement_combined')
fig.savefig('figures/prob_distribution.pdf', bbox_inches='tight')
fig.show()

## Per-paper breakdown

In [ ]:
paper_df = per_paper_metrics(
    extraction_df,
    ground_truth_df,
    judged_df=judged_df,
    strict_matching=STRICT_MATCHING,
    fuzzy_matching=FUZZY_MATCHING,
)
paper_df.sort_values('recall').round(3)